In [6]:
# ==== HMM on pre-breakout context for ORB ====
# Requirements: numpy, pandas, scipy, scikit-learn, hmmlearn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# --- sanity: require hmmlearn present
try:
    from hmmlearn.hmm import GaussianHMM
except ImportError as e:
    raise ImportError(
        "Missing dependency: hmmlearn. Install with: pip install hmmlearn"
    )

from sklearn.preprocessing import StandardScaler
from scipy.special import logsumexp

op_data = pd.read_csv(
    'C:/Users/User/Desktop/AlphaMath-QuantCore/Backtest/Csvs/NQ1!_15M_final_processed.csv'
)

# -----------------------------
# 0) Configuration
# -----------------------------
REQUIRED_COLS = [
    "overnight_range_over_daily_atr",
    "range_over_intraday_atr",
    "volume_over_daily_avg",
    "overnight_return_pct",
    "opening_range_stability",
]
POS_RATIO_COLS = [
    "overnight_range_over_daily_atr",
    "range_over_intraday_atr",
    "volume_over_daily_avg",
]
STABILITY_COL = "opening_range_stability"

K_RANGE = range(2, 6)  # try 2..5 states
COV_TYPE = "full"
RANDOM_STATE = 42
N_ITER = 1000
TOL = 1e-6

# -----------------------------
# 1) Prepare features (log / logit + standardize)
# -----------------------------
def _logit(x):
    return np.log(x / (1.0 - x))

def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def prepare_X(op_data: pd.DataFrame):
    df = op_data.copy()

    missing = set(REQUIRED_COLS) - set(df.columns)
    if missing:
        raise ValueError(f"op_data missing columns: {sorted(missing)}")

    # Coerce to float
    for c in REQUIRED_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Basic winsorization (helps stability with heavy tails)
    # Adjust quantiles as you like.
    WINS_Q = (0.005, 0.995)
    for c in REQUIRED_COLS:
        lo, hi = df[c].quantile(WINS_Q[0]), df[c].quantile(WINS_Q[1])
        df[c] = df[c].clip(lower=lo, upper=hi)

    # Positive ratios -> log
    for c in POS_RATIO_COLS:
        df[c] = df[c].clip(lower=np.finfo(float).eps)  # ensure >0
        df[c + "_log"] = np.log(df[c])

    # Stability (0,1) -> logit
    s = df[STABILITY_COL].clip(1e-6, 1 - 1e-6)
    df["stability_logit"] = _logit(s)

    # Overnight return stays as-is
    FEAT_COLS = [
        "overnight_return_pct",
        "overnight_range_over_daily_atr_log",
        "range_over_intraday_atr_log",
        "volume_over_daily_avg_log",
        "stability_logit",
    ]

    # Drop rows with any NaNs in the required feats
    df = df.dropna(subset=FEAT_COLS)
    X = df[FEAT_COLS].to_numpy().astype(float)

    # Standardize
    scaler = StandardScaler()
    Xz = scaler.fit_transform(X)

    return Xz, scaler, df.reset_index(drop=True), FEAT_COLS

# -----------------------------
# 2) Model selection via BIC
# -----------------------------
def count_params_gaussian_hmm_full(K: int, D: int) -> int:
    # startprob: K-1 free
    # transmat:  K*(K-1) free (each row sums to 1)
    # means:     K*D
    # covars:    K * D*(D+1)/2   (full covariance)
    return (K - 1) + K * (K - 1) + K * D + K * (D * (D + 1) // 2)

def fit_hmm_bic(Xz: np.ndarray,
                k_range=K_RANGE,
                covariance_type=COV_TYPE,
                random_state=RANDOM_STATE,
                n_iter=N_ITER,
                tol=TOL):
    D = Xz.shape[1]
    results = []
    best = None

    for K in k_range:
        model = GaussianHMM(
            n_components=K,
            covariance_type=covariance_type,
            n_iter=n_iter,
            tol=tol,
            random_state=random_state,
            verbose=False
        )
        model.fit(Xz)
        loglik = model.score(Xz)  # log-likelihood of the sequence
        n_params = count_params_gaussian_hmm_full(K, D)
        T = Xz.shape[0]
        bic = -2 * loglik + n_params * np.log(T)
        aic =  2 * n_params - 2 * loglik
        results.append({
            "K": K,
            "loglik": loglik,
            "params": n_params,
            "BIC": bic,
            "AIC": aic,
            "model": model
        })
        if best is None or bic < best["BIC"]:
            best = results[-1]

    summary = pd.DataFrame([{k: v for k, v in r.items() if k != "model"} for r in results]).sort_values("K")
    best_model = best["model"]
    return best_model, summary, results

# -----------------------------
# 3) Posteriors: smoothed + filtered, and Viterbi
# -----------------------------
def posteriors_and_viterbi(model: GaussianHMM, Xz: np.ndarray):
    # Smoothed posteriors from forward-backward
    logprob, gamma_smoothed = model.score_samples(Xz)  # (T,K)

    # Filtered posteriors via forward pass only (no "peeking")
    # Use model internals to get emission log-likelihoods per state/time
    log_B = model._compute_log_likelihood(Xz)  # (T,K)  (private but standard in hmmlearn)
    A = model.transmat_
    pi = model.startprob_

    T, K = log_B.shape
    log_alpha = np.empty((T, K))
    log_alpha[0] = np.log(pi + 1e-300) + log_B[0]
    for t in range(1, T):
        # log_alpha_t(k) = log_B_t(k) + logsum_j log_alpha_{t-1}(j) + log A_{j,k}
        log_alpha[t] = log_B[t] + logsumexp(log_alpha[t-1][:, None] + np.log(A + 1e-300), axis=0)

    # Normalize to probabilities per t
    gamma_filtered = np.exp(log_alpha - logsumexp(log_alpha, axis=1, keepdims=True))

    # Viterbi path (most likely state sequence)
    viterbi_states = model.predict(Xz)

    return gamma_smoothed, gamma_filtered, viterbi_states, logprob

# -----------------------------
# 4) Stationary distribution & expected durations
# -----------------------------
def stationary_dist(A: np.ndarray):
    # left eigenvector at eigenvalue 1
    w, v = np.linalg.eig(A.T)
    idx = np.argmin(np.abs(w - 1.0))
    vec = np.real(v[:, idx])
    pi = vec / vec.sum()
    # ensure nonnegative (numerical guard)
    pi = np.clip(pi, 0, None)
    if pi.sum() == 0:
        # fallback: uniform
        pi = np.ones_like(pi) / len(pi)
    else:
        pi = pi / pi.sum()
    return pi

def expected_durations(A: np.ndarray):
    # E[duration in state k] = 1 / (1 - A_kk)  (geometric)
    diag = np.clip(np.diag(A), 0, 0.999999)
    return 1.0 / (1.0 - diag)

# -----------------------------
# 5) Back-transform state means to original feature units
# -----------------------------
def backtransform_state_means(model: GaussianHMM,
                              scaler: StandardScaler,
                              feat_cols: list):
    # model.means_ are in z-space; bring them back to transformed X-space…
    means_z = model.means_  # (K,D)
    means_X = means_z * scaler.scale_ + scaler.mean_

    # …then invert per-feature transforms to original units for interpretability
    rows = []
    for k in range(means_X.shape[0]):
        row = {"state": k}
        for j, f in enumerate(feat_cols):
            val = means_X[k, j]
            if f.endswith("_log"):
                base = f[:-4]
                row[f] = val
                row[base] = float(np.exp(val))
            elif f == "stability_logit":
                row[f] = val
                row["opening_range_stability"] = float(_sigmoid(val))
            else:
                # overnight_return_pct
                row[f] = float(val)
        rows.append(row)

    df = pd.DataFrame(rows).set_index("state")
    # Order columns: original-unit view first
    ordered = [
        "overnight_return_pct",
        "overnight_range_over_daily_atr",
        "range_over_intraday_atr",
        "volume_over_daily_avg",
        "opening_range_stability",
        "overnight_range_over_daily_atr_log",
        "range_over_intraday_atr_log",
        "volume_over_daily_avg_log",
        "stability_logit",
    ]
    existing = [c for c in ordered if c in df.columns]
    return df[existing]

# -----------------------------
# 6) Run the whole pipeline on op_data
# -----------------------------
# Expecting op_data already in your workspace.
if "op_data" not in globals():
    raise RuntimeError("Please define a pandas DataFrame named `op_data` with the required columns before running.")

Xz, scaler, op_data_clean, FEAT_COLS = prepare_X(op_data)

best_model, bic_table, _ = fit_hmm_bic(Xz, k_range=K_RANGE, covariance_type=COV_TYPE,
                                       random_state=RANDOM_STATE, n_iter=N_ITER, tol=TOL)

print("\n=== BIC Model Selection ===")
print(bic_table.to_string(index=False))

K = best_model.n_components
print(f"\nChosen K (by BIC): {K}")

print("\n=== Start probabilities ===")
print(pd.Series(best_model.startprob_, index=[f"state_{k}" for k in range(K)]).to_string())

print("\n=== Transition matrix ===")
A = best_model.transmat_
A_df = pd.DataFrame(A, index=[f"from_{k}" for k in range(K)], columns=[f"to_{k}" for k in range(K)])
print(A_df.to_string())

# Expected durations
dur = expected_durations(A)
print("\nExpected duration (in observations) per state (geom):")
print(pd.Series(dur, index=[f"state_{k}" for k in range(K)]).round(2).to_string())

# Stationary dist
pi_star = stationary_dist(A)
print("\nStationary distribution (left eigenvector):")
print(pd.Series(pi_star, index=[f"state_{k}" for k in range(K)]).round(4).to_string())

# State means (original-unit interpretation and transformed)
state_means_bt = backtransform_state_means(best_model, scaler, FEAT_COLS)
print("\n=== State means (interpretable, back in original units; with transformed columns for reference) ===")
print(state_means_bt.round(4).to_string())

# Posteriors and Viterbi
gamma_smooth, gamma_filt, viterbi_states, total_loglik = posteriors_and_viterbi(best_model, Xz)
print(f"\nTotal log-likelihood (best model): {total_loglik:.2f}")

# Attach to op_data for later use
post_sm_df = pd.DataFrame(gamma_smooth, columns=[f"p_smoothed_state_{k}" for k in range(K)])
post_fl_df = pd.DataFrame(gamma_filt,   columns=[f"p_filtered_state_{k}" for k in range(K)])
vit_df      = pd.DataFrame({"viterbi_state": viterbi_states})

op_data_hmm = pd.concat([op_data_clean.reset_index(drop=True), post_sm_df, post_fl_df, vit_df], axis=1)

print("\n=== Head of state posteriors (filtered) and Viterbi ===")
print(op_data_hmm[[c for c in op_data_hmm.columns if c.startswith("p_filtered_state_")] + ["viterbi_state"]].head(10).round(4).to_string(index=False))

# If you want, you can keep op_data_hmm around for later mapping to outcomes:
#   - Use p_filtered_state_k at the decision time (no peeking) for calibration q_{k,±} or F_{k,±}(y)
#   - Use viterbi_state only for interpretation, not for probability/EV



=== BIC Model Selection ===
 K        loglik  params          BIC          AIC
 2 -12158.669817      43 24642.601889 24403.339635
 3 -11725.618524      68 23965.605263 23587.237047
 4 -11163.917360      95 23046.437375 22517.834720
 5 -11026.675006     124 22991.315583 22301.350012

Chosen K (by BIC): 5

=== Start probabilities ===
state_0     0.000000e+00
state_1    4.801084e-223
state_2     0.000000e+00
state_3     0.000000e+00
state_4     1.000000e+00

=== Transition matrix ===
            to_0          to_1          to_2          to_3      to_4
from_0  0.058744  3.231885e-01  7.608853e-03  1.416179e-01  0.468840
from_1  0.215300  2.181549e-01  8.109627e-02  5.214748e-58  0.485449
from_2  0.044368  2.001502e-63  9.183032e-01  1.754621e-02  0.019783
from_3  0.030967  1.942811e-01  1.877512e-01  3.332409e-01  0.253760
from_4  0.293282  2.339106e-01  8.465142e-51  1.023555e-02  0.462572

Expected duration (in observations) per state (geom):
state_0     1.06
state_1     1.28
state_2   

In [7]:
# =====================================================================
# HMM → Outcome Mapping (extended):
# - Breakout probability r_k
# - Direction mix g_{k,±}
# - Hit probability q_{k,d} (with Beta shrinkage + CrI)
# - One-inflated Beta per (state, direction) for EACH metric in:
#     ['breakout_success', 'mfe_to_crossback_ratio', 'mfe_to_halfstop_ratio']
# - Weighted summaries & live probability/EV helpers
# =====================================================================

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple, Iterable
from scipy.stats import beta as beta_dist
from scipy.special import betaln
from scipy import optimize

# -------------------------------
# 0) Utilities
# -------------------------------
def _find_state_cols(df_like) -> list:
    cols = [c for c in (df_like.columns if hasattr(df_like, "columns") else df_like.index) 
            if str(c).startswith("p_filtered_state_")]
    if not cols:
        raise ValueError("No filtered posterior columns found (expected 'p_filtered_state_*').")
    return sorted(cols, key=lambda x: int(str(x).split("_")[-1]))

def _soft_counts(df: pd.DataFrame, wcol: str, mask=None) -> float:
    if mask is not None:
        return float(df.loc[mask, wcol].sum())
    return float(df[wcol].sum())

def _weighted_mean_var(y, w, eps=1e-12):
    y = np.asarray(y, float); w = np.asarray(w, float)
    W = w.sum()
    if W <= eps:
        return np.nan, np.nan
    m = (w * y).sum() / W
    v = (w * (y - m)**2).sum() / max(W, eps)
    return m, v

def _weighted_quantile(y, w, qs: Iterable[float]):
    """
    Weighted quantiles on y with nonnegative weights w. qs in [0,1].
    """
    y = np.asarray(y, float); w = np.asarray(w, float)
    m = np.isfinite(y) & np.isfinite(w) & (w >= 0)
    y, w = y[m], w[m]
    if y.size == 0 or w.sum() <= 0:
        return [np.nan for _ in qs]
    idx = np.argsort(y)
    y_sorted = y[idx]
    w_sorted = w[idx]
    cw = np.cumsum(w_sorted)
    cw /= cw[-1]
    return [float(y_sorted[np.searchsorted(cw, q, side="left")]) for q in qs]

def _fit_beta_mle_weighted(y, w, start=None, bounds=(1e-3, 1e3)):
    """
    Weighted MLE for Beta(alpha,beta) on y in (0,1). Returns (alpha, beta).
    Falls back to method-of-moments if necessary.
    """
    y = np.asarray(y, float); w = np.asarray(w, float)
    m, v = _weighted_mean_var(y, w)
    if not np.isfinite(m) or not np.isfinite(v) or v <= 1e-10 or m <= 0 or m >= 1:
        return 2.0, 5.0  # benign fallback
    t = m * (1 - m) / max(v, 1e-12) - 1
    a0 = max(m * t, 1e-3); b0 = max((1 - m) * t, 1e-3)
    if start is None or not np.all(np.isfinite(start)):
        start = np.array([a0, b0], float)

    wy_logy = (w * np.log(y)).sum()
    wy_log1y = (w * np.log(1 - y)).sum()
    W = w.sum()

    def nll(theta):
        a, b = np.maximum(theta, 1e-8)
        return -(a - 1) * wy_logy - (b - 1) * wy_log1y + W * betaln(a, b)

    res = optimize.minimize(nll, x0=start, method="L-BFGS-B",
                            bounds=[bounds, bounds], options={"maxiter": 1000})
    if not res.success or not np.all(np.isfinite(res.x)):
        return float(a0), float(b0)
    a, b = res.x
    return float(np.clip(a, 1e-3, 1e6)), float(np.clip(b, 1e-3, 1e6))

@dataclass
class OneInflatedBeta:
    pi: float     # point mass at 1
    alpha: float  # Beta for non-hits on (0,1)
    beta: float

    def p_ge_tau(self, tau: float) -> float:
        tau = float(np.clip(tau, 0.0, 1.0))
        if tau >= 1.0:  return float(self.pi)
        if tau <= 0.0:  return 1.0
        return float(self.pi + (1.0 - self.pi) * (1.0 - beta_dist.cdf(tau, self.alpha, self.beta)))

    def expected_value(self) -> float:
        if np.isfinite(self.alpha) and np.isfinite(self.beta):
            return float(self.pi + (1.0 - self.pi) * (self.alpha / (self.alpha + self.beta)))
        return float(self.pi)

# -------------------------------
# 1) r_k and g_{k,±}
# -------------------------------
def estimate_breakout_and_direction(op_data_hmm: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    state_cols = _find_state_cols(op_data_hmm)
    df = op_data_hmm.copy()
    if "break_direction" not in df.columns:
        raise ValueError("Missing 'break_direction'.")

    df["break_direction"] = df["break_direction"].fillna(0).round().clip(-1, 1).astype(int)

    rows_r, rows_g = [], []
    for k, wcol in enumerate(state_cols):
        Wk = _soft_counts(df, wcol)
        Wk_br = _soft_counts(df, wcol, mask=(df["break_direction"] != 0))
        r_k = Wk_br / Wk if Wk > 0 else np.nan

        Wk_up = _soft_counts(df, wcol, mask=(df["break_direction"] == +1))
        Wk_dn = _soft_counts(df, wcol, mask=(df["break_direction"] == -1))
        # Dirichlet(1,1) smoothing
        g_up = (Wk_up + 1.0) / (Wk_br + 2.0) if Wk_br > 0 else np.nan
        g_dn = (Wk_dn + 1.0) / (Wk_br + 2.0) if Wk_br > 0 else np.nan

        rows_r.append({"state": k, "soft_weight": Wk, "soft_breakouts": Wk_br, "r_breakout": r_k})
        rows_g.append({"state": k, "soft_breakouts": Wk_br, "g_up": g_up, "g_down": g_dn})

    return (pd.DataFrame(rows_r).set_index("state"),
            pd.DataFrame(rows_g).set_index("state"))

# -------------------------------
# 2) q_{k,d} on breakout_success (target hit prob)
# -------------------------------
def estimate_hit_probabilities(op_data_hmm: pd.DataFrame,
                               alpha0: float = 1.0, beta0: float = 1.0) -> pd.DataFrame:
    state_cols = _find_state_cols(op_data_hmm)
    df = op_data_hmm.copy()
    if "break_direction" not in df.columns or "breakout_success" not in df.columns:
        raise ValueError("Need 'break_direction' and 'breakout_success'.")
    df["Y"] = df["breakout_success"].clip(0.0, 1.0)

    rows = []
    for k, wcol in enumerate(state_cols):
        for d in (-1, +1):
            mask_d = (df["break_direction"] == d)
            W_trials = _soft_counts(df, wcol, mask=mask_d)
            W_succ   = _soft_counts(df, wcol, mask=(mask_d & (df["Y"] >= 1.0 - 1e-12)))
            a_post = alpha0 + W_succ
            b_post = beta0  + (W_trials - W_succ)
            q_hat  = a_post / (a_post + b_post) if (a_post + b_post) > 0 else np.nan
            lo = beta_dist.ppf(0.025, a_post, b_post) if (a_post>0 and b_post>0) else np.nan
            hi = beta_dist.ppf(0.975, a_post, b_post) if (a_post>0 and b_post>0) else np.nan

            rows.append({
                "state": k, "direction": d,
                "soft_trials": W_trials, "soft_successes": W_succ,
                "q_hat": q_hat, "q_lo95": lo, "q_hi95": hi,
                "alpha_post": a_post, "beta_post": b_post
            })
    return pd.DataFrame(rows).set_index(["state", "direction"])

# -------------------------------
# 3) One-inflated Beta per metric (generic)
# -------------------------------
def fit_one_inflated_beta_for_metrics(op_data_hmm: pd.DataFrame,
                                      metrics = ("breakout_success",
                                                 "mfe_to_crossback_ratio",
                                                 "mfe_to_halfstop_ratio")) -> Dict[str, pd.DataFrame]:
    """
    For each metric in metrics (assumed in [0,1]), fit one-inflated Beta per (state, direction).
    Returns dict: metric -> DataFrame indexed by (state, direction) with columns
      ['soft_trials','soft_nonhits','pi','alpha','beta','E_Y','w_mean','q50','q75','q90','q95'].
    """
    state_cols = _find_state_cols(op_data_hmm)
    df = op_data_hmm.copy()
    if "break_direction" not in df.columns:
        raise ValueError("Missing 'break_direction'.")

    out = {}
    for metric in metrics:
        if metric not in df.columns:
            raise ValueError(f"Missing outcome column '{metric}'.")
        ycol = metric
        df[ycol] = pd.to_numeric(df[ycol], errors="coerce").clip(0.0, 1.0)

        rows = []
        for k, wcol in enumerate(state_cols):
            for d in (-1, +1):
                msk = (df["break_direction"] == d)
                w_all = df.loc[msk, wcol].to_numpy(float)
                y_all = df.loc[msk, ycol].to_numpy(float)

                W_trials = float(np.nansum(w_all))
                if W_trials <= 0:
                    rows.append({"state": k, "direction": d,
                                 "soft_trials": 0.0, "soft_nonhits": 0.0,
                                 "pi": np.nan, "alpha": np.nan, "beta": np.nan,
                                 "E_Y": np.nan, "w_mean": np.nan,
                                 "q50": np.nan, "q75": np.nan, "q90": np.nan, "q95": np.nan})
                    continue

                # Atom at 1 (exact hits)
                hit_mask = (y_all >= 1.0 - 1e-12)
                W_hit = float(np.nansum(w_all[hit_mask]))
                pi = W_hit / W_trials

                # Fit Beta for non-hits (0<y<1)
                nh_mask = (~hit_mask) & (y_all > 0) & (y_all < 1)
                y_nh = y_all[nh_mask]
                w_nh = w_all[nh_mask]
                if y_nh.size >= 3 and np.nansum(w_nh) > 0:
                    a, b = _fit_beta_mle_weighted(y_nh, w_nh)
                else:
                    a, b = np.nan, np.nan

                # Weighted descriptive stats (overall, not just non-hits)
                w_mean, _ = _weighted_mean_var(y_all, w_all)
                q50, q75, q90, q95 = _weighted_quantile(y_all, w_all, qs=[0.50, 0.75, 0.90, 0.95])

                # Expected value of one-inflated Beta (fallback to pi if a/b missing)
                if np.isfinite(a) and np.isfinite(b):
                    EY = float(pi + (1.0 - pi) * (a / (a + b)))
                else:
                    EY = float(pi)

                rows.append({
                    "state": k, "direction": d,
                    "soft_trials": W_trials,
                    "soft_nonhits": float(np.nansum(w_nh)),
                    "pi": float(pi), "alpha": float(a), "beta": float(b),
                    "E_Y": EY, "w_mean": float(w_mean),
                    "q50": q50, "q75": q75, "q90": q90, "q95": q95
                })

        out[metric] = pd.DataFrame(rows).set_index(["state", "direction"])
    return out

# -------------------------------
# 4) Mixture helpers (any metric) + EV for target-hit
# -------------------------------
def p_tau_kd(oneinfl_df: pd.DataFrame, k: int, d: int, tau: float) -> float:
    row = oneinfl_df.loc[(k, d)]
    pi, a, b = row["pi"], row["alpha"], row["beta"]
    if not np.isfinite(pi):
        return np.nan
    if not (np.isfinite(a) and np.isfinite(b)):
        # fallback: atom-only (mass at 1)
        return float(pi) if tau >= 1.0 else (1.0 if tau <= 0.0 else float(pi))
    return OneInflatedBeta(float(pi), float(a), float(b)).p_ge_tau(tau)

def p_tau_mixture_for_metric(op_row: pd.Series,
                             params_per_metric: Dict[str, pd.DataFrame],
                             metric: str,
                             tau: float,
                             direction: int) -> float:
    state_cols = _find_state_cols(op_row.to_frame().T)
    df_params = params_per_metric[metric]
    s = 0.0; any_added = False
    for k, wcol in enumerate(state_cols):
        p_k = float(op_row[wcol])
        if p_k <= 0 or not np.isfinite(p_k): 
            continue
        pkd = p_tau_kd(df_params, k, int(direction), tau)
        if np.isfinite(pkd):
            s += p_k * pkd
            any_added = True
    return float(s) if any_added else np.nan

def expected_value_mixture_for_metric(op_row: pd.Series,
                                      params_per_metric: Dict[str, pd.DataFrame],
                                      metric: str,
                                      direction: int) -> float:
    state_cols = _find_state_cols(op_row.to_frame().T)
    df_params = params_per_metric[metric]
    s = 0.0; any_added = False
    for k, wcol in enumerate(state_cols):
        p_k = float(op_row[wcol])
        if p_k <= 0 or not np.isfinite(p_k): 
            continue
        row = df_params.loc[(k, int(direction))]
        pi, a, b = row["pi"], row["alpha"], row["beta"]
        if not np.isfinite(pi): 
            continue
        if np.isfinite(a) and np.isfinite(b):
            s += p_k * OneInflatedBeta(float(pi), float(a), float(b)).expected_value()
        else:
            s += p_k * float(pi)  # fallback
        any_added = True
    return float(s) if any_added else np.nan

def ev_from_prob(p_hit: float, G: float = 1.0, L: float = 1.0, cost: float = 0.0) -> float:
    if not np.isfinite(p_hit): 
        return np.nan
    return p_hit * G - (1.0 - p_hit) * L - cost

# -------------------------------
# 5) Runner that prints and returns everything
# -------------------------------
def run_estimations_extended(op_data_hmm: pd.DataFrame,
                             alpha0: float = 1.0, beta0: float = 1.0
                            ) -> Dict[str, object]:
    # Breakout & direction
    df_breakout_prob, df_dir_mix = estimate_breakout_and_direction(op_data_hmm)

    # Target-hit (breakout_success)
    df_hit_prob = estimate_hit_probabilities(op_data_hmm, alpha0=alpha0, beta0=beta0)

    # One-inflated Beta for all metrics
    per_metric_params = fit_one_inflated_beta_for_metrics(
        op_data_hmm,
        metrics=("breakout_success", "mfe_to_crossback_ratio", "mfe_to_halfstop_ratio")
    )

    # ---- Prints
    print("\n=== Breakout probability by state (r_k) ===")
    print(df_breakout_prob[["soft_weight","soft_breakouts","r_breakout"]].round(4).to_string())

    print("\n=== Direction mix given breakout (g_{k,±}) ===")
    print(df_dir_mix[["soft_breakouts","g_up","g_down"]].round(4).to_string())

    print("\n=== Target-hit probability q_{k,d} on 'breakout_success' (95% CrI) ===")
    print(df_hit_prob[["soft_trials","soft_successes","q_hat","q_lo95","q_hi95"]].round(4).to_string())

    for metric, dfp in per_metric_params.items():
        print(f"\n=== One-inflated Beta params & weighted summaries for '{metric}' ===")
        cols = ["soft_trials","soft_nonhits","pi","alpha","beta","E_Y","w_mean","q50","q75","q90","q95"]
        print(dfp[cols].round(4).to_string())

    return {
        "breakout_prob": df_breakout_prob,
        "direction_mix": df_dir_mix,
        "hit_prob": df_hit_prob,                # on breakout_success
        "oneinfl_per_metric": per_metric_params # dict: metric -> df
    }

# -------------------------------
# 6) Examples (uncomment to use)
# -------------------------------
results = run_estimations_extended(op_data_hmm)
params = results["oneinfl_per_metric"]

# Live probability that we reach the FULL target (tau=1.0) on current row i:
i = 0
d = int(op_data_hmm.loc[i, "break_direction"])  # use +1/-1 at actual break time
p_hit_full = p_tau_mixture_for_metric(op_data_hmm.loc[i], params, metric="breakout_success", tau=1.0, direction=d)
print(f"\nRow {i} — P(hit target) = {p_hit_full:.3f}")

# Live probability that MFE-before-crossback reaches at least 60% of target:
p_cb_60 = p_tau_mixture_for_metric(op_data_hmm.loc[i], params, metric="mfe_to_crossback_ratio", tau=0.60, direction=d)
print(f"Row {i} — P(MFE-to-crossback ≥ 0.60) = {p_cb_60:.3f}")

# Expected fractions (for sizing/partials) on the two new metrics:
e_cb = expected_value_mixture_for_metric(op_data_hmm.loc[i], params, metric="mfe_to_crossback_ratio", direction=d)
e_hs = expected_value_mixture_for_metric(op_data_hmm.loc[i], params, metric="mfe_to_halfstop_ratio",   direction=d)
print(f"Row {i} — E[MFE-to-crossback] = {e_cb:.3f},  E[MFE-to-halfstop] = {e_hs:.3f}")

# EV with 1:1 RRR from P(hit)
ev = ev_from_prob(p_hit_full, G=1.0, L=1.0, cost=0.0)
print(f"Row {i} — EV (R) = {ev:.3f}")



=== Breakout probability by state (r_k) ===
       soft_weight  soft_breakouts  r_breakout
state                                         
0         314.9721        314.9721      1.0000
1         350.2556        350.2556      1.0000
2         537.0943        537.0943      1.0000
3          88.7716         87.7716      0.9887
4         636.9064        636.9064      1.0000

=== Direction mix given breakout (g_{k,±}) ===
       soft_breakouts    g_up  g_down
state                                
0            314.9721  0.5422  0.4578
1            350.2556  0.5291  0.4709
2            537.0943  0.4964  0.5036
3             87.7716  0.4808  0.5192
4            636.9064  0.5196  0.4804

=== Target-hit probability q_{k,d} on 'breakout_success' (95% CrI) ===
                 soft_trials  soft_successes   q_hat  q_lo95  q_hi95
state direction                                                     
0     -1            144.1088         71.6570  0.4973  0.4166  0.5780
       1            170.8633     